# Tutorial: Using an LLM to Improve Data Quality (First Iteration)

This notebook is a practical, **first-iteration** walkthrough of a real business use case: taking a messy dataset, exploring it to surface quality issues, using a Large Language Model (LLM) to propose fixes, and writing results back out. Machine Learning (ML) and Artificial Intelligence (AI) cover a huge landscape far more than any single notebook can capture so I am intentionally **not** repeating material you can find in depth elsewhere. Instead, this focuses on one concrete, end-to-end workflow you can copy and extend.

## Quick AI/ML Overview

The AI/ML landscape offers different tools for different problems:

- **Rule-based systems**: Deterministic logic (e.g., if/else rules, regex patterns) for known patterns. Great for consistent, well-understood errors.
  - *When to use*: You know exactly what's wrong and how to fix it (e.g., "all phone numbers missing area codes should prepend '555'")
  - *Limitations*: Brittle when patterns vary; maintenance overhead grows as edge cases accumulate

- **Classical ML**: Statistical models that learn patterns from labeled data (e.g., regression, decision trees, random forests). Strong when you have clean labels and stable data distributions.
  - *When to use*: You have labeled training data and patterns are consistent but too complex for simple rules
  - *Limitations*: Requires feature engineering; struggles with novel patterns; needs retraining as data drifts

- **Deep Learning**: Neural networks that learn hierarchical representations from large datasets (e.g., CNNs for images, RNNs for sequences). High performance but often data-hungry.
  - *When to use*: Complex pattern recognition with abundant training data (computer vision, speech recognition)
  - *Limitations*: Requires substantial compute and data; often a "black box"; prone to overfitting on small datasets

- **LLMs (Generative AI)**: Transformer-based models pre-trained on massive text corpora that can interpret context, follow instructions, and generate coherent language. Useful for messy, semi-structured data where rules are brittle or expensive to maintain.
  - *When to use*: Ambiguous or context-dependent corrections; varied data formats; you want to iterate quickly without collecting training data
  - *Limitations*: Non-deterministic outputs; can "hallucinate" or make confident errors; higher per-call cost than rule-based approaches; requires prompt engineering and validation

**The practical takeaway**: Start simple. Use rules when you can and don't underestimate how far you can get with basic mathmatical & statistical approches, escalate to ML when patterns are learnable from data and you have labeled datasets to train on, and reach for LLMs when you need flexible reasoning over _language_ or when building training datasets would be prohibitively expensive.


In this tutorial, we use an LLM as a **data curation assistant** to suggest corrections in row-level records. The goal is to show how you might do an initial, high-leverage pass on data quality before scaling or automating the approach.

__Further resources:__
- https://www.tiktok.com/@mlgonzo1 - Our very own Mike LG, fountain of knowledge!

- https://www.3blue1brown.com/topics/neural-networks An EXCELLENT video series over several hours on Neural networks and Large launguge models

- https://www.kaggle.com/ - Oepn source datasets and community for all aspects of machine learning.

- https://www.youtube.com/watch?v=Z78zbnLlPUA&list=PLQVvvaa0QuDdttJXlLtAJxJetJcqmqlQq Great computer vision and image processing course.

- https://www.youtube.com/watch?v=OGxgnH8y2NM&list=PLQVvvaa0QuDfKTOs3Keq_kaG2P55YRn5v Same author as above - deep dive into machine learning methods.

- For cutting edge, state of the art type stuff your going to have to dig into the academic literature - searching for recent review style papers on google scholar is a great place to start!







## Step 1: Import libraries

We load the Python packages needed for the workflow: Pandas for data handling, `dotenv` for configuration, the Azure OpenAI client for LLM calls, and `json_repair` for handling imperfect JSON outputs. Keeping imports centralized makes dependencies explicit and reproducible.

In [ ]:
import json

import pandas as pd

from dotenv import dotenv_values
from openai import AzureOpenAI
from json_repair import repair_json


## Step 2: Load configuration from `.env`

Configuration (file paths, model settings, keys) is stored in `.env` so the notebook stays portable and secrets are not hard-coded. We print values to confirm the environment is loaded correctly before touching data or APIs.


In [ ]:
config = dotenv_values(".env")

for k, v in config.items():
    print(f"key: {k}, value: {v}")

## Step 3: Load the dataset and take a first look

We read the input table into a DataFrame, print its shape, and preview the head. This is a quick sanity check to verify file access and understand the basic scale of the dataset.


In [ ]:
df = pd.read_csv(config["input_table_path"])
print(f"Total number of rows and columns = {df.shape}")
df.head()


## Step 4: List all columns

Printing column names is a lightweight way to scan the schema and spot obvious issues (misspellings, unexpected fields, or missing columns) before deeper analysis.


In [ ]:
for col in df.columns:
    print(col)

## Step 5: Narrow to the columns we care about

We select a subset of fields that matter for the cleaning exercise. This keeps the test set small, focused, and easier to reason about during debugging. No I have not Idea what Text_10 relates to either!


In [ ]:
cols = [
    "Id",
    "Type",
    "Code",
    "Name",
    "Address1",
    "Address2",
    "Town",
    "County",
    "Country",
    "PostCode",
    "Telephone",
    "FAX",
    "Notes",
    "Last_Update",
    "Last_User",
    "EMailNumber",
    "NI_Building_Type",
    "NI_Building_Category",
    "Latitude",
    "Longitude",
]

df_sm = df[cols]
print(f"Total number of rows and columns = {df.shape}")
df_sm.head(10)

## Step 6: Quickly filter for rows with phone numbers

Ok so some immidiate curiositys here, weird that many contry feilds are missing, seems an easy-ish fix right? Last user varies between names and email addresses - it shouldnt as there is a seperate feild for that! Inconsistant capitalisation all over. longitude and lattitude 0.0, 0.0? Thats actually a legit location in the middle of the ocean called null island - these should actually be Null values!

Lets start by taking a deeper look at the telephone column by filtering rows where `Telephone` is not null

In [ ]:
df_with_telephone = df_sm[df_sm['Telephone'].notna()]
df_with_telephone.head(10)

## Step 7: Build a small test set

Interesting! Those happen to be what 3 word references and not telephone numbers at all! Lets define a list of IDs and pull just those rows into a test DataFrame. This is a common practice for LLM workflows: iterate on prompts with a controlled sample before scaling to the full dataset.


In [ ]:
test_ids = [
    22132,
    10132,
    12359,
    1089,
    10761,
    2347,
    2145,
    2431,
    676,
    1184,
]

df_test_set = df_sm[df_sm['Id'].isin(test_ids)]
print(f"Total number of rows and columns = {df_test_set.shape}")
df_test_set.head(11)

## Step 8: Inspect one representative row

Lets further condense this to a single test row to play with -  convert a single record to a dictionary and print it key-by-key. This makes it easy to read and understand the specific data issues we want the LLM to fix.


In [ ]:
first_row_dict = df_test_set.loc[df_test_set['Id'] == 10132].iloc[0].to_dict()
for k, v in first_row_dict.items():
    print(f"{k}: {v}")

## Step 9: Configure and sanity-check the Azure OpenAI client

Ok, now we've got something we could actually send to an LLM over and API (almost), lets set up the LLM object and check it works. We load model settings from the config, create the Azure OpenAI client, and run a simple test prompt. This confirms that credentials and deployment details are correct before we send real data.


In [ ]:
endpoint = config["endpoint"]
model_name = config["model_name"]
deployment = config["deployment"]

subscription_key = config["subscription_key"]
api_version = config["api_version"]

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

response = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant.",
        },
        {
            "role": "user",
            "content": "What is the capital of France?",
        }
    ],
    max_tokens=4096,
    temperature=1.0,
    top_p=1.0,
    model=deployment
)

print(response.choices[0].message.content)

## Step 10: Wrap the LLM call in a helper function
Ok all good! We don't really want to write out the response code each time so lets create a reusable function that accepts a system prompt and a user prompt. This keeps the notebook clean and allows us to tweak prompts without duplicating API call code.


In [ ]:
def curate_call(system_prompt, user_prompt):

    response = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": user_prompt,
            }
        ],
        max_tokens=4096,
        temperature=1.0,
        top_p=1.0,
        model=deployment
    )
    
    print(response.choices[0].message.content)
    return response.choices[0].message.content

## Step 11: Convert the sample row to a string for prompting

The LLM expects text input, so we stringify the row dictionary so it's a string of a dictionary. Printing the result confirms the exact string that will be sent to the model.


In [ ]:
bad_data = str(first_row_dict)
print(type(bad_data))
print(bad_data)

## Step 12: Draft a first-pass system prompt

This initial prompt tells the model its role and the kind of fixes to apply. It is intentionally simple so we can see where it fails before tightening the instructions.


In [ ]:
system_prompt = """
    You are a data expert. Your role is to clense a row of data which will be given to you as a string of a JSON object.
    You should check:
    - The values make sense in contex of the key.

    Where the values do not make sense in context of the key, you should suggest a fix, you can use any of the data inth the JSON object
    to make this fix.
"""

user_prompt = f"The row to fix is: {bad_data}"

## Step 13: Run the initial prompt

We call the LLM with the basic prompt to see how it responds. This output informs how we refine the instructions for more structured, reliable results.


In [ ]:
curate_call(system_prompt, user_prompt)

## Step 14: Create a detailed, constrained prompt
Great that makes sense... But we cant actually use that in an automated way, it's a verbose response - how would we parse it back apart and figure out what to write back to the database?
Here we provide a richer prompt with examples and a strict output format. This is critical when you need deterministic, machine-parseable JSON from an LLM.


In [ ]:
system_prompt = """
You are a data expert. Your role is to clense a row of data which will be given to you as a string of a JSON object.
You should check:
- The values make sense in contex of the key.
- we know there are systemic shifts of data across columns in a row, for example an address having more than 2 lines so the city might
  be found in the county column.
- Invalid or null or nan values should ALWAYS be updated to null.
- check formats,  what 3 word references should always be seperated by periods.

Where the values do not make sense in context of the key, you should suggest a fix, you can use any of the data inth the JSON object
to make this fix.



**EXAMPLES**
Input from user:
{
'Id': 10132,
'Type': nan,
'Code': 'E2341360',
'Name': 'Aintree GSM-R 4399',
'Address1': 'Off A59 Ormskirk Road',
'Address2': 'through Asda car park',
'Town': 'Aintree',
'County': 'Liverpool',
'Country': nan,
'PostCode': 'L10 3LN',
'Telephone': 'cult.ample,broad',
'FAX': nan,
'Notes': 
'Enter Asda car park drive past the petrol station and thefirst  Manweb substation turn right and take rough track under the rail overbridge and meter cubcile is in fenceline next to the track access gate on the LHS\r\nMEx Emlite IP 192.168.8.15 Port 19740 fitted 10/11/2025', 
'Last_Update': '2025-11-10 17:16:44.0000000',
'Last_User': 'Terry Bleasdale',
'EMailNumber': nan,
'NI_Building_Type': 0,
'NI_Building_Category': 0,
'Latitude': 0.0,
'Longitude': 0.0
}

GOOD Cleansed output:
{
'Id': 10132,
'Type': None,
'Code': 'E2341360',
'Name': 'Aintree GSM-R 4399',
'Address1': 'Off A59 Ormskirk Road',
'Address2': 'Aintree',
'Town': 'Liverpool',
'County': 'Merseyside',
'Country': England,
'PostCode': 'L10 3LN',
'Telephone': None,
'what_3_words': 'cult.ample.broad',
'FAX': None,
'Notes': 
'Enter Asda car park drive past the petrol station and thefirst  Manweb substation turn right and take rough track under the rail overbridge and meter cubcile is in fenceline next to the track access gate on the LHS\r\nMEx Emlite IP 192.168.8.15 Port 19740 fitted 10/11/2025', 
'Last_Update': '2025-11-10 17:16:44.0000000',
'Last_User': 'Terry Bleasdale',
'EMailNumber': None,
'NI_Building_Type': None,
'NI_Building_Category': None,
'Latitude': None,
'Longitude': None,
'changes_applied' : [
"empty or invalid values converted to None",
"Address2 corrected from a site note to the town which was found in the County column",
"Conunty incorrect value moved to correct column Town",
"Country populated correctly",
"what 3 words reference not found in input schema but value found in Telephone, updated accordingly",
]
}

**OUTPUT FORMAT**
The output schema should **ALWAYS** be a valid JSON object and follow this schema:
{
'Id': number,
'Type' : string,
'Code': string,
'Name': string,
'Address1': string,
'Address2': string,
'Town': string,
'County': string,
'Country': string,
'PostCode': string,
'Telephone': number,
'what_3_words': string,
'FAX': number,
'Notes': string,
'Last_Update' datetime,
'Last_User': string,
'EMailNumber': string,
'NI_Building_Type': string,
'NI_Building_Category': string,
'Latitude': number,
'Longitude': number,
'changes_applied': List of strings,
}
"""

user_prompt = f"Do not give a description or explanation, only a valid Json object as your response,do not include ANY newline charachters! The row to fix is: {bad_data}"

## Step 15: Execute the refined prompt

We run the LLM again with the improved instructions and capture the response. This response is intended to be structured JSON that can be parsed downstream.


In [ ]:
response = curate_call(system_prompt, user_prompt)

## Step 16: Parse and repair the LLM response

Much better! Note some of the following prompt engineering techniques::
- We gave it a role with a list of objectives.
- We capitilised or otherwise highlighted important points.
- We used repitition around key points - JSON object.
- We gave a good example - on shot prompting.

But LLM outputs are not always valid JSON. This next helper function strips code fences, normalizes `None` to `null`, and uses a repair step before parsing.


In [ ]:
def transform_string_to_dict(response):
    """parse LLM responses"""
    # Remove markdown code fences
    stripped_string = (
        response.strip()
        .removeprefix("```json")
        .removeprefix("```")
        .removesuffix("```")
        .strip()
    )
    
    # Replace Python None with JSON null
    cleaned_string = stripped_string.replace("None", "null")
    
    # Try to repair and parse as JSON
    try:
        good_json_string = repair_json(cleaned_string)
        response_dict = json.loads(good_json_string)
        return response_dict
    except json.JSONDecodeError as e:
        print(f"JSON parsing error: {e}")
        print(f"Problematic string: {cleaned_string[:500]}")  # Print first 500 chars for debugging
        raise

## Step 17: Inspect the parsed output

We parse the response and print each field with its type. This verifies that the LLM output is valid JSON and that fields are coerced into usable Python types.


In [ ]:
response_dict = transform_string_to_dict(response)

print(type(response_dict))
for k, v in response_dict.items():
    print(f"key: {k},        value: {v}         value type: {type(v)} ")

## Step 18: Scale the curation to the test set
Ok so prehaps we've cheated a little as the one-shot example we gave in the prompt IS the data we gave it. Lets see how that performs against our test set.
We loop over the test DataFrame, send each row to the LLM, parse the response, and collect cleaned records. This simulates batch processing on a small sample.


In [ ]:
data = []

for index, row in df_test_set.iterrows():
    row_dict = row.to_dict()
    user_prompt = f"Do not give a description or explanation, only a valid JSON object as your response. The row to fix is: {str(row_dict)}"
    response = curate_call(system_prompt, user_prompt)
    response_dict = transform_string_to_dict(response)
    data.append(response_dict)

## Step 19: Convert cleaned records back to a DataFrame
We rebuild a DataFrame from the curated rows so we can review the results with standard Pandas tooling.


In [ ]:
# Convert list of dictionaries back to DataFrame
df_curated = pd.DataFrame(data)

df_curated.head(10)

## Step 20: Write results to disk

We save a CSV to the output path.


In [ ]:
df_curated.to_csv(config["output_table_path"], index=False)


# Next steps:

Once you've worked through the notebook:
  - How can we tweek the prompt for improvments?
  - Explore the data set further to uncover more issues, think about how you can use the process to have the LLM actually do this for you!
  - Think about transparency here, LLMs hallucinate and get things wrong, we can manually check 100 entries but the full data set is actually 26k rows. What could we do to mitigate this risk?
  - Think about technical limitations: 
    - This is currently a seriel process.
    - How deal with Failed API calls?
    - How make idempotent as a process?
  - Think about cost, How would you give a cost estimate of this? How would that break down?
  - Think about scalability - we're using a single prompt here to fix multiple issues, at what point will that start to breakdown and we need an alternative approch?
  - BIG ONE: We've solved some issues here using GenAI but some of them could be solved with regex methods which would be faster, cheaper and more transparent. Where is the dividing line? when should we NOT use AI or ML methods?